In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import os

# ===================== 1. PathandParametersSet =====================
output_dir = "D:/MLY/ABO3/Machine learning process/code-22/S1-pcc/"
os.makedirs(output_dir, exist_ok=True)

# [YouCanModify yourself]OnlyPreserve PCC LargeAtThisValuedata
PCC_THRESHOLD = 0.8  

# ===================== 2. Readdata =====================
data = pd.read_csv("D:/MLY/ABO3/Machine learning process/code-22/S1-pcc/Qf.csv")
X = data.drop(columns=["targ"])
features = X
cols = features.columns

# ===================== 3. Calculate PCC and pValueMatrix =====================
pcc_matrix = features.corr(method="pearson")

p_values = np.zeros_like(pcc_matrix)
for i in range(len(cols)):
    for j in range(len(cols)):
        corr, p = pearsonr(features.iloc[:, i], features.iloc[:, j])
        p_values[i, j] = p

p_matrix = pd.DataFrame(p_values, index=cols, columns=cols)

# ===================== 4. Plot heatmap: TopThreeAnglePCC | BottomThreeAnglepValue =====================
mask = np.triu(np.ones_like(pcc_matrix))
combined = np.where(mask, pcc_matrix, p_matrix)
combined_df = pd.DataFrame(combined, index=cols, columns=cols)

plt.figure(figsize=(12, 10))
sns.heatmap(
    combined_df,
    cmap="RdBu_r",
    vmin=-1, vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    annot_kws={"size": 8}
)
plt.title(f"PCC (Upper) | p-value (Lower) | PCC > {PCC_THRESHOLD}", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "PCC_Significance_Heatmap.png"), dpi=300, bbox_inches="tight")
plt.close()

# ===================== 5. ExportFilterBack CSV(Core functionality) =====================
# StoreFormat: Feature1 | Feature2 | PCC | pValue
result_rows = []

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):  # OnlyGetTopThreeAngle(NotRepeat)
        pcc_val = pcc_matrix.iloc[i, j]
        p_val = p_matrix.iloc[i, j]
        
        # OnlyPreserve PCC LargeAtSetThresholddata
        if pcc_val > PCC_THRESHOLD:
            result_rows.append([
                cols[i],
                cols[j],
                round(pcc_val, 4),
                round(p_val, 6)
            ])

# BuildDataFrameAndSave
result_df = pd.DataFrame(
    result_rows,
    columns=["VerticalCoordinateFeature", "HorizontalCoordinateFeature", "PCCValue", "pValue"]
)

# ===================== Fixed: FileNameNotUsable > Symbol =====================
csv_path = os.path.join(output_dir, f"PCC_above_{PCC_THRESHOLD}.csv")
result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

# ===================== [Newly added]BatchScatter plot drawing =====================
# CreateScatter plotSaveFileFolder
scatter_dir = os.path.join(output_dir, "dataInformation")
os.makedirs(scatter_dir, exist_ok=True)

# ReadHighRelatedFeatureCorrectAndPlot
for idx, row in result_df.iterrows():
    feat1 = row["VerticalCoordinateFeature"]
    feat2 = row["HorizontalCoordinateFeature"]
    pcc = row["PCCValue"]
    p_val = row["pValue"]
    
    # CreateCanvas
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=features[feat1], y=features[feat2], color='#2E86AB', alpha=0.7)
    
    # LabelRelatedSystemCountandSignificance
    plt.title(f"{feat1} vs {feat2}\nPCC = {pcc:.3f} | p = {p_val:.2e}", fontsize=12)
    plt.xlabel(feat1)
    plt.ylabel(feat2)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    # SaveImage(SafetyFileName)
    save_name = f"scatter_{feat1}_{feat2}.png".replace(":", "").replace("\\", "").replace("/", "")
    plt.savefig(os.path.join(scatter_dir, save_name), dpi=300, bbox_inches='tight')
    plt.close()

# ===================== Completed =====================
print("✅ HeatmapAlreadySaveTo: ", output_dir)
print("✅ CSVAlreadySaveTo: ", csv_path)
print(f"✅ AlreadyFilterOutput PCC > {PCC_THRESHOLD} FeatureCombination")
print(f"✅ Scatter plotAlreadySaveToFileFolder: [dataInformation]")